# Biped Balance Policy Training (mjlab + RSL-RL PPO on Colab)

This notebook trains the biped's balance policy using **mjlab** (MuJoCo Warp) and **RSL-RL**'s PPO on a Colab GPU runtime.

**Requires a GPU runtime**: Runtime > Change runtime type > GPU.

This notebook expects the upload bundle described in `docs/colab_upload_manifest.md` (created in a later sub-task) to already be present in the working directory before running the cells below.

In [ ]:
!nvidia-smi

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU detected - set Runtime > Change runtime type > GPU"
print(f"CUDA available: {torch.cuda.is_available()}, device: {torch.cuda.get_device_name(0)}")

In [ ]:
# The bundle's requirements-colab.txt must already be present in the
# working directory (see the upload-bundle cell below).
!pip install -r requirements-colab.txt

In [ ]:
# Verify the upload bundle matches docs/colab_upload_manifest.md before
# continuing. The manifest documents the exact minimal set of files this
# notebook expects in the working directory (biped_warp.xml, meshes/,
# mjlab_biped/, requirements-colab.txt) - see that file for the full
# layout and rationale.
import os

expected_paths = [
    "models/mjcf/biped_warp.xml",
    "meshes/stl",
    "mjlab_biped",
    "requirements-colab.txt",
]
missing = [p for p in expected_paths if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(
        f"Upload bundle is incomplete, missing: {missing}. "
        f"See docs/colab_upload_manifest.md for the full expected bundle."
    )
print("Bundle looks complete:", os.listdir("."))


In [ ]:
import mjlab
import mujoco_warp
import rsl_rl
import importlib.metadata


def _version(pkg_name, module):
    try:
        return module.__version__
    except AttributeError:
        return importlib.metadata.version(pkg_name)


print("mjlab version:", _version("mjlab", mjlab))
print("mujoco_warp version:", _version("mujoco-warp", mujoco_warp))
print("rsl_rl version:", _version("rsl-rl-lib", rsl_rl))

## Register the biped task and run a smoke test

`mjlab_biped/mjlab_task.py` is the first module in this bundle that imports
the *real* `mjlab` package -- it translates the already-tested pure-Python
specs in the rest of `mjlab_biped/` into real mjlab manager-API objects and
registers `"Mjlab-Biped-Balance-v0"`. Several attribute-name guesses in
that file are explicitly UNVERIFIED (see `docs/mjlab_adapter_notes.md`) --
this smoke test is designed to fail FAST and CHEAP on any of them, before
any real (multi-hour) training run starts.


In [ ]:
import mjlab_biped.mjlab_task as biped_task

print(f"Registered task: {biped_task.TASK_ID}")


In [ ]:
# Minimal smoke test: build a tiny (num_envs=1) real mjlab env, reset it,
# step it once with a zero action, and print observation shapes. This is
# expected to catch any of the UNVERIFIED attribute-name guesses in
# mjlab_task.py immediately (AttributeError) rather than deep into a real
# training run. See docs/mjlab_adapter_notes.md for what to check first
# if this cell raises an error.
import torch

smoke_env_cfg = biped_task.make_biped_env_cfg(num_envs=1)
# NOTE: the exact mjlab env construction call (ManagerBasedRlEnv(cfg=...)
# vs a factory function) is itself unverified -- adjust this line if it
# doesn't match mjlab's real env-construction API.
from mjlab.envs import ManagerBasedRlEnv

env = ManagerBasedRlEnv(cfg=smoke_env_cfg)
obs, extras = env.reset()
print("actor obs shape:", obs["actor"].shape)
print("critic obs shape:", obs["critic"].shape)
assert torch.isfinite(obs["actor"]).all(), "actor obs contains NaN/Inf"
assert torch.isfinite(obs["critic"]).all(), "critic obs contains NaN/Inf"

zero_action = torch.zeros(1, 6, device=obs["actor"].device)
obs, reward, terminated, truncated, extras = env.step(zero_action)
print("post-step reward:", reward)
print("post-step actor obs finite:", torch.isfinite(obs["actor"]).all().item())
print("\nSmoke test passed. Safe to proceed to training below.")


## Train

Uses the `train` console script installed by the `mjlab` package itself
(confirmed real, see `docs/mjlab_adapter_notes.md`). Start with a small
`num_envs` and `max_iterations` for a real (not just smoke-test) training
check before scaling up -- e.g. a few hundred iterations to see the reward
curve start moving in a sane direction, well short of the frozen
`max_iterations=1500` budget in `mjlab_biped/rl_cfg.py`.

The exact `--env.*`/`--agent.*` override flag names are UNVERIFIED (see
notes doc) -- if a flag below is rejected, run `!train --help` first to
see the real flag names for this installed mjlab version, then adjust.


In [ ]:
!train Mjlab-Biped-Balance-v0 --env.scene.num-envs 64 --agent.max-iterations 200


## Eval (`mjlab play`)

Renders rollouts from a saved checkpoint. Fill in the checkpoint path
printed by the training cell above.


In [ ]:
checkpoint_path = ""  # TODO: fill in from the training cell's logged output
assert checkpoint_path, "Set checkpoint_path to a real checkpoint file before running this cell."
!play Mjlab-Biped-Balance-v0 --checkpoint-file {checkpoint_path}
